# Sleep Disorder Prediction from NHANES Questionnaire Features

## Goal
Identify which NHANES questionnaire variables are most useful for predicting the target variable `sleep_disorder`.

## Important constraints
- We want **quiz-eligible** features, not just maximum predictive performance.
- We exclude:
  - target leakage
  - medication / diagnosis / RXD variables
  - survey design variables
  - lab / exam / technical derived variables
  - features that cannot realistically be asked in an online quiz
- The target `sleep_disorder` is treated as a **broad proxy**, not a clinical diagnosis.

## Outputs
- Univariate ranking of questionnaire features
- Model-based feature importance ranking
- Shortlist of quiz candidates

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import re
import json
import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency, pointbiserialr

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyClassifier

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

In [5]:
DATA_PATH = Path("../data/processed/nhanes_merged_adults_final.csv")
OUTPUT_DIR = Path("../data/processed/model_outputs_sleep_disorder")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "sleep_disorder"
RANDOM_STATE = 42
TEST_SIZE = 0.20

print("DATA_PATH:", DATA_PATH.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

DATA_PATH: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\nhanes_merged_adults_final.csv
OUTPUT_DIR: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_sleep_disorder


In [6]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("First 10 columns:")
print(df.columns[:10].tolist())

Shape: (7437, 877)
First 10 columns:
['SEQN', 'age_years', 'income_poverty_ratio', 'mec_exam_weight', 'interview_weight', 'survey_psu', 'survey_stratum', 'gender', 'ethnicity', 'education']


In [7]:
assert TARGET in df.columns, f"{TARGET} not found in dataframe."

print(df[TARGET].value_counts(dropna=False).sort_index())
print()
print("Target prevalence:")
print(df[TARGET].value_counts(normalize=True, dropna=False).sort_index().round(4))

sleep_disorder
0    5034
1    2403
Name: count, dtype: int64

Target prevalence:
sleep_disorder
0    0.6769
1    0.3231
Name: proportion, dtype: float64


In [8]:
overview = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_pct": df.isna().mean().values * 100,
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns]
}).sort_values(["missing_pct", "n_unique"], ascending=[False, True])

overview.head(30)

,feature,dtype,missing_pct,n_unique
166,mcq149___menstrual_periods_started_yet?,float64,100.000000,0
167,mcq151___age_in_years_at_first_menstrual_period,float64,100.000000,0
294,smq621___cigarettes_smoked_in_entire_life,float64,100.000000,0
295,smd630___age_first_smoked_whole_cigarette,float64,100.000000,0
434,LBXIGG_cytomegalovirus_cmv_igg,float64,100.000000,0
435,LBXIGM_cytomegalovirus_cmv_igm,float64,100.000000,0
436,LBXIGGA_cytomegalovirus_cmv_igg_avidity,float64,100.000000,0
357,medication_21,str,99.973107,2
358,medication_22,str,99.973107,2
356,medication_20,str,99.959661,3


In [9]:
# Columns used directly in / very close to target construction
LEAKAGE_COLS = [
    TARGET,
    "slq040___how_often_do_you_snort_or_stop_breathing",
    "slq050___ever_told_doctor_had_trouble_sleeping?",
    "rxd_disease_list",
]

# Prefixes that are likely questionnaire-style and potentially quiz-eligible
QUESTIONNAIRE_PREFIXES = (
    "slq", "sld", "dpq", "paq", "huq", "mcq", "bpq", "kiq", "cdq", "smq", "alq"
)

# Patterns we generally want to exclude
EXCLUDE_PATTERNS = [
    r"^seqn$",
    r"cluster",
    r"stratum",
    r"weight",
    r"sample_weight",
    r"survey_",
    r"^rxd",
    r"medication",
    r"drug",
    r"icd",
    r"diagnosis",
    r"lab",
    r"exam",
    r"fasting",
    r"liver_",
    r"bmi$",
    r"waist",
    r"hip",
    r"height",
    r"weight_kg",
    r"cholesterol",
    r"creatin",
    r"ferritin",
    r"vitamin",
    r"iron",
    r"protein",
    r"carbs",
    r"fat$",
    r"calories",
    r"pulse_",
    r"fatigue_binary",
    r"fatigue_score",
]

# Optional: columns that are questionnaire-based but probably too close to outcome/general morbidity
MANUAL_DROP_COLS = [
    # add manually after inspection if needed
]

In [10]:
def has_allowed_prefix(col: str, prefixes=QUESTIONNAIRE_PREFIXES) -> bool:
    col_low = col.lower()
    return col_low.startswith(prefixes)

def matches_any_pattern(col: str, patterns) -> bool:
    col_low = col.lower()
    return any(re.search(p, col_low) for p in patterns)

def is_binary_series(s: pd.Series) -> bool:
    vals = set(pd.Series(s.dropna()).unique())
    return vals.issubset({0, 1}) and len(vals) <= 2

def get_example_values(s: pd.Series, n=5):
    vals = s.dropna().unique()[:n]
    return list(vals)

In [11]:
candidate_cols = []

for col in df.columns:
    if col == TARGET:
        continue
    if col in LEAKAGE_COLS:
        continue
    if not has_allowed_prefix(col):
        continue
    if matches_any_pattern(col, EXCLUDE_PATTERNS):
        continue
    if col in MANUAL_DROP_COLS:
        continue
    candidate_cols.append(col)

print("Number of initial candidate questionnaire features:", len(candidate_cols))
print(candidate_cols[:50])

Number of initial candidate questionnaire features: 112
['alq111___ever_had_a_drink_of_any_kind_of_alcohol', 'alq121___past_12_mo_how_often_drink_alcoholic_bev', 'alq130___avg_#_alcoholic_drinks/day___past_12_mos', 'alq142___#_days_have_4_or_5_drinks/past_12_mos', 'alq270___#_times_4_5_drinks_in_2hrs/past_12_mos', 'alq280___#_times_8+_drinks_in_1_day/past_12_mos', 'alq290___#_times_12+_drinks_in_1_day/past_12_mos', 'alq151___ever_have_4/5_or_more_drinks_every_day?', 'alq170___past_30_days_#_times_4_5_drinks_on_an_oc', 'bpq020___ever_told_you_had_high_blood_pressure', 'bpq030___told_had_high_blood_pressure___2+_times', 'bpq040a___taking_prescription_for_hypertension', 'bpq050a___now_taking_prescribed_medicine_for_hbp', 'bpq100d___now_taking_prescribed_medicine', 'cdq001___sp_ever_had_pain_or_discomfort_in_chest', 'cdq002___sp_get_it_walking_uphill_or_in_a_hurry', 'cdq003___during_an_ordinary_pace_on_level_ground', 'cdq004___if_so_does_sp_continue_or_slow_down', 'cdq005___does_standing_r

In [12]:
audit_rows = []

for col in candidate_cols:
    s = df[col]
    audit_rows.append({
        "feature": col,
        "dtype": str(s.dtype),
        "missing_pct": round(s.isna().mean() * 100, 2),
        "n_unique": s.nunique(dropna=True),
        "example_values": str(get_example_values(s, n=5)),
        "is_binary_like": is_binary_series(s),
        "keep_for_model": True,
        "drop_reason": ""
    })

audit_df = pd.DataFrame(audit_rows).sort_values(
    ["missing_pct", "n_unique", "feature"],
    ascending=[False, True, True]
)

audit_df.head(100)

,feature,dtype,missing_pct,n_unique,example_values,is_binary_like,keep_for_model,drop_reason
57,mcq149___menstrual_periods_started_yet?,float64,100.00,0,[],True,True,
58,mcq151___age_in_years_at_first_menstrual_period,float64,100.00,0,[],True,True,
111,smq621___cigarettes_smoked_in_entire_life,float64,100.00,0,[],True,True,
78,mcq230c___what_kind_of_cancer_third_mention,float64,99.93,4,"[np.float64(20.0), np.float64(23.0), np.float6...",False,True,
26,cdq009g___pain_in_left_arm,float64,99.85,1,[np.float64(7.0)],False,True,
22,cdq009c___pain_in_neck,float64,99.84,1,[np.float64(3.0)],False,True,
20,cdq009a___pain_in_right_arm,float64,99.81,1,[np.float64(1.0)],True,True,
77,mcq230b___what_kind_of_cancer_second_mention,float64,99.61,15,"[np.float64(37.0), np.float64(16.0), np.float6...",False,True,
21,cdq009b___pain_in_right_chest,float64,99.41,1,[np.float64(2.0)],False,True,
24,cdq009e___pain_in_lower_sternum,float64,99.23,1,[np.float64(5.0)],False,True,


In [13]:
# Basic automated pruning rules
MAX_MISSING_PCT = 60.0
MIN_NON_NULL = 500
MIN_VARIANCE_UNIQUE = 2

filtered_audit_df = audit_df.copy()

filtered_audit_df.loc[filtered_audit_df["missing_pct"] > MAX_MISSING_PCT, "keep_for_model"] = False
filtered_audit_df.loc[filtered_audit_df["missing_pct"] > MAX_MISSING_PCT, "drop_reason"] = "too_missing"

for idx, row in filtered_audit_df.iterrows():
    col = row["feature"]
    non_null = df[col].notna().sum()
    if non_null < MIN_NON_NULL:
        filtered_audit_df.at[idx, "keep_for_model"] = False
        filtered_audit_df.at[idx, "drop_reason"] = "too_few_non_null"
    elif row["n_unique"] < MIN_VARIANCE_UNIQUE:
        filtered_audit_df.at[idx, "keep_for_model"] = False
        filtered_audit_df.at[idx, "drop_reason"] = "constant_or_near_constant"

filtered_audit_df["keep_for_model"].value_counts()

keep_for_model
True     61
False    51
Name: count, dtype: int64

In [14]:
# Add features here after you inspect the audit table.
# Example:
MANUAL_DROP_AFTER_AUDIT = [
    # "huq051___#times_receive_healthcare_over_past_year",
    # "huq090___seen_mental_health_professional/past_yr",
]

filtered_audit_df.loc[
    filtered_audit_df["feature"].isin(MANUAL_DROP_AFTER_AUDIT),
    ["keep_for_model", "drop_reason"]
] = [False, "manual_drop"]

final_features = filtered_audit_df.loc[filtered_audit_df["keep_for_model"], "feature"].tolist()

print("Final feature count:", len(final_features))
print(final_features[:50])

Final feature count: 61
['paq670___days_moderate_recreational_activities', 'paq625___number_of_days_moderate_work', 'cdq001___sp_ever_had_pain_or_discomfort_in_chest', 'cdq010___shortness_of_breath_on_stairs/inclines', 'alq170___past_30_days_#_times_4_5_drinks_on_an_oc', 'alq142___#_days_have_4_or_5_drinks/past_12_mos', 'alq130___avg_#_alcoholic_drinks/day___past_12_mos', 'alq151___ever_have_4/5_or_more_drinks_every_day?', 'alq121___past_12_mo_how_often_drink_alcoholic_bev', 'kiq046___leak_urine_during_nonphysical_activities', 'kiq480___how_many_times_urinate_in_night?', 'kiq044___urinated_before_reaching_the_toilet?', 'kiq042___leak_urine_during_physical_activities?', 'kiq005___how_often_have_urinary_leakage?', 'dpq040___feeling_tired_or_having_little_energy', 'alq111___ever_had_a_drink_of_any_kind_of_alcohol', 'kiq022___ever_told_you_had_weak/failing_kidneys?', 'kiq026___ever_had_kidney_stones?', 'mcq160a___ever_told_you_had_arthritis', 'mcq160b___ever_told_you_had_congestive_heart_f

In [15]:
final_audit_path = OUTPUT_DIR / "sleep_disorder_feature_audit_final.csv"
filtered_audit_df.to_csv(final_audit_path, index=False)
print("Saved:", final_audit_path.resolve())

Saved: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_sleep_disorder\sleep_disorder_feature_audit_final.csv


In [16]:
model_df = df[[TARGET] + final_features].copy()

print("Model dataframe shape:", model_df.shape)
print("Target missing:", model_df[TARGET].isna().sum())

Model dataframe shape: (7437, 62)
Target missing: 0


In [17]:
X = model_df[final_features].copy()
y = model_df[TARGET].copy()

# Keep only rows with non-missing target
mask = y.notna()
X = X.loc[mask].copy()
y = y.loc[mask].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target distribution:")
print(y_train.value_counts(normalize=True).round(4))

Train shape: (5949, 61)
Test shape: (1488, 61)
Train target distribution:
sleep_disorder
0    0.6769
1    0.3231
Name: proportion, dtype: float64


In [18]:
def cramers_v(x: pd.Series, y: pd.Series) -> float:
    table = pd.crosstab(x, y)
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    chi2 = chi2_contingency(table)[0]
    n = table.values.sum()
    r, k = table.shape
    denom = min(k - 1, r - 1)
    if denom == 0 or n == 0:
        return np.nan
    return np.sqrt((chi2 / n) / denom)

def safe_pointbiserial(x: pd.Series, y: pd.Series):
    valid = x.notna() & y.notna()
    if valid.sum() < 30:
        return np.nan, np.nan
    try:
        r, p = pointbiserialr(x[valid], y[valid])
        return r, p
    except Exception:
        return np.nan, np.nan

def infer_feature_type(series: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(series):
        nunique = series.nunique(dropna=True)
        if nunique <= 10:
            return "categorical"
        return "numeric"
    return "categorical"

In [20]:
univariate_rows = []

for col in final_features:
    s = X_train[col]
    feature_type = infer_feature_type(s)

    row = {
        "feature": col,
        "feature_type": feature_type,
        "missing_pct_train": round(s.isna().mean() * 100, 2),
        "n_unique_train": s.nunique(dropna=True),
        "association_metric": None,
        "association_value": np.nan,
        "abs_association": np.nan,
        "p_value": np.nan,
        "n_train_non_null": int(s.notna().sum())
    }

    if feature_type == "categorical":
        valid = s.notna() & y_train.notna()
        if valid.sum() >= 30:
            try:
                cv = cramers_v(s[valid], y_train[valid])
                table = pd.crosstab(s[valid], y_train[valid])
                p_val = chi2_contingency(table)[1] if table.shape[0] > 1 and table.shape[1] > 1 else np.nan
                row["association_metric"] = "cramers_v"
                row["association_value"] = cv
                row["abs_association"] = abs(cv) if pd.notna(cv) else np.nan
                row["p_value"] = p_val
            except Exception:
                pass
    else:
        r, p = safe_pointbiserial(s, y_train)
        row["association_metric"] = "pointbiserial"
        row["association_value"] = r
        row["abs_association"] = abs(r) if pd.notna(r) else np.nan
        row["p_value"] = p

    univariate_rows.append(row)

univariate_df = pd.DataFrame(univariate_rows).sort_values(
    ["abs_association", "p_value"],
    ascending=[False, True]
)

univariate_df.head(50)


,feature,feature_type,missing_pct_train,n_unique_train,association_metric,association_value,abs_association,p_value,n_train_non_null
60,huq051___#times_receive_healthcare_over_past_year,categorical,0.00,10,cramers_v,0.315201,0.315201,1.749666e-121,5949
14,dpq040___feeling_tired_or_having_little_energy,categorical,12.81,6,cramers_v,0.291063,0.291063,9.354725e-93,5187
18,mcq160a___ever_told_you_had_arthritis,categorical,6.44,3,cramers_v,0.285244,0.285244,4.571333e-99,5566
3,cdq010___shortness_of_breath_on_stairs/inclines,categorical,43.69,3,cramers_v,0.271130,0.271130,3.347455e-54,3350
42,huq090___seen_mental_health_professional/past_yr,categorical,0.00,3,cramers_v,0.254319,0.254319,2.808160e-84,5949
2,cdq001___sp_ever_had_pain_or_discomfort_in_chest,categorical,43.69,3,cramers_v,0.251564,0.251564,9.203970e-47,3350
59,huq010___general_health_condition,categorical,0.00,7,cramers_v,0.236729,0.236729,5.675892e-69,5949
47,mcq366b___doctor_told_to_increase_exercise,categorical,0.00,3,cramers_v,0.231306,0.231306,7.673916e-70,5949
40,bpq020___ever_told_you_had_high_blood_pressure,categorical,0.00,3,cramers_v,0.226528,0.226528,5.135945e-67,5949
58,slq030___how_often_do_you_snore?,categorical,0.00,6,cramers_v,0.215314,0.215314,1.591744e-57,5949


In [21]:
univariate_path = OUTPUT_DIR / "sleep_disorder_univariate_ranking.csv"
univariate_df.to_csv(univariate_path, index=False)
print("Saved:", univariate_path.resolve())

Saved: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_sleep_disorder\sleep_disorder_univariate_ranking.csv


In [22]:
numeric_features = []
categorical_features = []

for col in final_features:
    if infer_feature_type(X_train[col]) == "numeric":
        numeric_features.append(col)
    else:
        categorical_features.append(col)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 6
Categorical features: 55


In [23]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

In [24]:
baseline_model = DummyClassifier(strategy="most_frequent")

baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

print("Baseline precision:", precision_score(y_test, baseline_pred, zero_division=0))
print("Baseline recall:", recall_score(y_test, baseline_pred, zero_division=0))
print("Baseline f1:", f1_score(y_test, baseline_pred, zero_division=0))

Baseline precision: 0.0
Baseline recall: 0.0
Baseline f1: 0.0


In [25]:
logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

In [26]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

In [27]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}

logreg_cv = cross_validate(logreg_pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
rf_cv = cross_validate(rf_pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

cv_summary = pd.DataFrame({
    "model": ["logreg", "random_forest"],
    "roc_auc_mean": [logreg_cv["test_roc_auc"].mean(), rf_cv["test_roc_auc"].mean()],
    "avg_precision_mean": [logreg_cv["test_avg_precision"].mean(), rf_cv["test_avg_precision"].mean()],
    "f1_mean": [logreg_cv["test_f1"].mean(), rf_cv["test_f1"].mean()],
    "precision_mean": [logreg_cv["test_precision"].mean(), rf_cv["test_precision"].mean()],
    "recall_mean": [logreg_cv["test_recall"].mean(), rf_cv["test_recall"].mean()],
}).sort_values("avg_precision_mean", ascending=False)

cv_summary

,model,roc_auc_mean,avg_precision_mean,f1_mean,precision_mean,recall_mean
0,logreg,0.789819,0.656604,0.616223,0.564452,0.678473
1,random_forest,0.794305,0.656453,0.614838,0.589185,0.643090


In [28]:
best_pipeline = logreg_pipeline  # change to logreg_pipeline if logreg performs better
best_model_name = "logreg"

best_pipeline.fit(X_train, y_train)

test_proba = best_pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.50).astype(int)

print("Best model:", best_model_name)
print("ROC-AUC:", round(roc_auc_score(y_test, test_proba), 4))
print("Average Precision:", round(average_precision_score(y_test, test_proba), 4))
print("Precision:", round(precision_score(y_test, test_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, test_pred, zero_division=0), 4))
print("F1:", round(f1_score(y_test, test_pred, zero_division=0), 4))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))
print()
print(classification_report(y_test, test_pred, zero_division=0))

Best model: logreg
ROC-AUC: 0.7873
Average Precision: 0.6582
Precision: 0.5801
Recall: 0.6778
F1: 0.6251

Confusion Matrix:
[[771 236]
 [155 326]]

              precision    recall  f1-score   support

           0       0.83      0.77      0.80      1007
           1       0.58      0.68      0.63       481

    accuracy                           0.74      1488
   macro avg       0.71      0.72      0.71      1488
weighted avg       0.75      0.74      0.74      1488



In [29]:
perm = permutation_importance(
    best_pipeline,
    X_test,
    y_test,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring="average_precision",
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

perm_df.head(30)

,feature,importance_mean,importance_std
14,dpq040___feeling_tired_or_having_little_energy,0.044996,0.005032
42,huq090___seen_mental_health_professional/past_yr,0.033736,0.004847
60,huq051___#times_receive_healthcare_over_past_year,0.030185,0.005827
58,slq030___how_often_do_you_snore?,0.028695,0.003781
18,mcq160a___ever_told_you_had_arthritis,0.013104,0.005053
47,mcq366b___doctor_told_to_increase_exercise,0.008818,0.002471
40,bpq020___ever_told_you_had_high_blood_pressure,0.004383,0.001877
43,mcq010___ever_been_told_you_have_asthma,0.003724,0.001622
2,cdq001___sp_ever_had_pain_or_discomfort_in_chest,0.003446,0.001733
56,smq020___smoked_at_least_100_cigarettes_in_life,0.003324,0.000964


# Sleep-focused only run

This section restricts the feature space to sleep-relevant and quiz-eligible variables only.

## Purpose
Compare a broad questionnaire model (Set 1) against a sleep-focused model (Set 2).

## Principle
We intentionally exclude:
- healthcare utilization proxies
- broad morbidity proxies
- post-hoc disease burden markers
- medication / diagnosis variables

We keep only:
- sleep-related questionnaire items
- a few plausible supporting risk/symptom questions

In [30]:
SLEEP_FOCUSED_FEATURES = [
    # Core sleep-related questionnaire items
    "slq030___how_often_do_you_snore?",
    "sld012___sleep_hours___weekdays_or_workdays",
    "sld013___sleep_hours___weekends",
    "slq300___usual_sleep_time_on_weekdays_or_workdays",
    "slq310___usual_wake_time_on_weekdays_or_workdays",
    "slq320___usual_sleep_time_on_weekends",
    "slq330___usual_wake_time_on_weekends",

    # Daytime symptom / functional consequence
    "dpq040___feeling_tired_or_having_little_energy",

    # Plausible supportive risk factors / related symptoms
    "bpq020___ever_told_you_had_high_blood_pressure",
    "cdq010___shortness_of_breath_on_stairs/inclines",
    "mcq010___ever_been_told_you_have_asthma",
    "mcq160p___ever_told_you_had_copd_emphysema",

    # Optional lifestyle/context signals that may still be quiz-eligible
    "paq605___vigorous_work_activity",
    "paq665___moderate_recreational_activities",
    "alq111___ever_had_a_drink_of_any_kind_of_alcohol",
    "alq170___past_30_days_#_times_4_5_drinks_on_an_oc",
]

In [31]:
sleep_focused_available = [c for c in SLEEP_FOCUSED_FEATURES if c in df.columns]
sleep_focused_missing = [c for c in SLEEP_FOCUSED_FEATURES if c not in df.columns]

print("Available sleep-focused features:", len(sleep_focused_available))
print(sleep_focused_available)
print()
print("Missing requested features:", len(sleep_focused_missing))
print(sleep_focused_missing)

Available sleep-focused features: 16
['slq030___how_often_do_you_snore?', 'sld012___sleep_hours___weekdays_or_workdays', 'sld013___sleep_hours___weekends', 'slq300___usual_sleep_time_on_weekdays_or_workdays', 'slq310___usual_wake_time_on_weekdays_or_workdays', 'slq320___usual_sleep_time_on_weekends', 'slq330___usual_wake_time_on_weekends', 'dpq040___feeling_tired_or_having_little_energy', 'bpq020___ever_told_you_had_high_blood_pressure', 'cdq010___shortness_of_breath_on_stairs/inclines', 'mcq010___ever_been_told_you_have_asthma', 'mcq160p___ever_told_you_had_copd_emphysema', 'paq605___vigorous_work_activity', 'paq665___moderate_recreational_activities', 'alq111___ever_had_a_drink_of_any_kind_of_alcohol', 'alq170___past_30_days_#_times_4_5_drinks_on_an_oc']

Missing requested features: 0
[]


In [32]:
sleep_focus_audit_rows = []

for col in sleep_focused_available:
    s = df[col]
    sleep_focus_audit_rows.append({
        "feature": col,
        "dtype": str(s.dtype),
        "missing_pct": round(s.isna().mean() * 100, 2),
        "n_unique": s.nunique(dropna=True),
        "example_values": str(get_example_values(s, n=5)),
        "is_binary_like": is_binary_series(s),
    })

sleep_focus_audit_df = pd.DataFrame(sleep_focus_audit_rows).sort_values(
    ["missing_pct", "n_unique", "feature"],
    ascending=[False, True, True]
)

sleep_focus_audit_df

,feature,dtype,missing_pct,n_unique,example_values,is_binary_like
9,cdq010___shortness_of_breath_on_stairs/inclines,float64,43.83,3,"[np.float64(1.0), np.float64(2.0), np.float64(...",False
15,alq170___past_30_days_#_times_4_5_drinks_on_an_oc,float64,35.35,23,"[np.float64(0.0), np.float64(30.0), np.float64...",False
7,dpq040___feeling_tired_or_having_little_energy,float64,13.03,6,"[np.float64(0.0), np.float64(2.0), np.float64(...",False
14,alq111___ever_had_a_drink_of_any_kind_of_alcohol,float64,12.75,2,"[np.float64(1.0), np.float64(2.0)]",False
11,mcq160p___ever_told_you_had_copd_emphysema,float64,6.20,3,"[np.float64(2.0), np.float64(1.0), np.float64(...",False
2,sld013___sleep_hours___weekends,float64,0.86,24,"[np.float64(8.0), np.float64(13.0), np.float64...",False
1,sld012___sleep_hours___weekdays_or_workdays,float64,0.71,24,"[np.float64(7.5), np.float64(8.0), np.float64(...",False
3,slq300___usual_sleep_time_on_weekdays_or_workdays,str,0.65,65,"['22:00', '00:00', '23:00', '08:00', '01:00']",False
4,slq310___usual_wake_time_on_weekdays_or_workdays,str,0.62,113,"['05:30', '08:00', '06:30', '09:00', '14:35']",False
5,slq320___usual_sleep_time_on_weekends,str,0.59,59,"['23:00', '03:00', '21:00', '22:00', '00:00']",False


In [33]:
SLEEP_FOCUS_MANUAL_DROP = [
    # optional manually drop after inspection
]

SLEEP_FOCUS_MAX_MISSING_PCT = 60.0

sleep_focused_final_features = [
    c for c in sleep_focused_available
    if c not in SLEEP_FOCUS_MANUAL_DROP
    and df[c].isna().mean() * 100 <= SLEEP_FOCUS_MAX_MISSING_PCT
]

print("Final sleep-focused features:", len(sleep_focused_final_features))
print(sleep_focused_final_features)

Final sleep-focused features: 16
['slq030___how_often_do_you_snore?', 'sld012___sleep_hours___weekdays_or_workdays', 'sld013___sleep_hours___weekends', 'slq300___usual_sleep_time_on_weekdays_or_workdays', 'slq310___usual_wake_time_on_weekdays_or_workdays', 'slq320___usual_sleep_time_on_weekends', 'slq330___usual_wake_time_on_weekends', 'dpq040___feeling_tired_or_having_little_energy', 'bpq020___ever_told_you_had_high_blood_pressure', 'cdq010___shortness_of_breath_on_stairs/inclines', 'mcq010___ever_been_told_you_have_asthma', 'mcq160p___ever_told_you_had_copd_emphysema', 'paq605___vigorous_work_activity', 'paq665___moderate_recreational_activities', 'alq111___ever_had_a_drink_of_any_kind_of_alcohol', 'alq170___past_30_days_#_times_4_5_drinks_on_an_oc']


In [34]:
sleep_focus_df = df[[TARGET] + sleep_focused_final_features].copy()

mask = sleep_focus_df[TARGET].notna()
X_sf = sleep_focus_df.loc[mask, sleep_focused_final_features].copy()
y_sf = sleep_focus_df.loc[mask, TARGET].astype(int).copy()

X_train_sf, X_test_sf, y_train_sf, y_test_sf = train_test_split(
    X_sf, y_sf,
    test_size=TEST_SIZE,
    stratify=y_sf,
    random_state=RANDOM_STATE
)

print("Sleep-focused train shape:", X_train_sf.shape)
print("Sleep-focused test shape:", X_test_sf.shape)
print("Train target distribution:")
print(y_train_sf.value_counts(normalize=True).round(4))

Sleep-focused train shape: (5949, 16)
Sleep-focused test shape: (1488, 16)
Train target distribution:
sleep_disorder
0    0.6769
1    0.3231
Name: proportion, dtype: float64


In [35]:
sf_univariate_rows = []

for col in sleep_focused_final_features:
    s = X_train_sf[col]
    feature_type = infer_feature_type(s)

    row = {
        "feature": col,
        "feature_type": feature_type,
        "missing_pct_train": round(s.isna().mean() * 100, 2),
        "n_unique_train": s.nunique(dropna=True),
        "association_metric": None,
        "association_value": np.nan,
        "abs_association": np.nan,
        "p_value": np.nan,
        "n_train_non_null": int(s.notna().sum())
    }

    if feature_type == "categorical":
        valid = s.notna() & y_train_sf.notna()
        if valid.sum() >= 30:
            try:
                cv = cramers_v(s[valid], y_train_sf[valid])
                table = pd.crosstab(s[valid], y_train_sf[valid])
                p_val = chi2_contingency(table)[1] if table.shape[0] > 1 and table.shape[1] > 1 else np.nan
                row["association_metric"] = "cramers_v"
                row["association_value"] = cv
                row["abs_association"] = abs(cv) if pd.notna(cv) else np.nan
                row["p_value"] = p_val
            except Exception:
                pass
    else:
        r, p = safe_pointbiserial(s, y_train_sf)
        row["association_metric"] = "pointbiserial"
        row["association_value"] = r
        row["abs_association"] = abs(r) if pd.notna(r) else np.nan
        row["p_value"] = p

    sf_univariate_rows.append(row)

sf_univariate_df = pd.DataFrame(sf_univariate_rows).sort_values(
    ["abs_association", "p_value"],
    ascending=[False, True]
)

sf_univariate_df

,feature,feature_type,missing_pct_train,n_unique_train,association_metric,association_value,abs_association,p_value,n_train_non_null
7,dpq040___feeling_tired_or_having_little_energy,categorical,12.81,6,cramers_v,0.291063,0.291063,9.354725e-93,5187
9,cdq010___shortness_of_breath_on_stairs/inclines,categorical,43.69,3,cramers_v,0.271130,0.271130,3.347455e-54,3350
8,bpq020___ever_told_you_had_high_blood_pressure,categorical,0.00,3,cramers_v,0.226528,0.226528,5.135945e-67,5949
0,slq030___how_often_do_you_snore?,categorical,0.00,6,cramers_v,0.215314,0.215314,1.591744e-57,5949
11,mcq160p___ever_told_you_had_copd_emphysema,categorical,6.44,3,cramers_v,0.181734,0.181734,1.207192e-40,5566
3,slq300___usual_sleep_time_on_weekdays_or_workdays,categorical,0.62,64,cramers_v,0.145843,0.145843,4.537005e-06,5912
4,slq310___usual_wake_time_on_weekdays_or_workdays,categorical,0.61,107,cramers_v,0.144723,0.144723,1.134996e-01,5913
10,mcq010___ever_been_told_you_have_asthma,categorical,0.00,3,cramers_v,0.144117,0.144117,1.477888e-27,5949
6,slq330___usual_wake_time_on_weekends,categorical,0.59,64,cramers_v,0.135568,0.135568,3.098512e-04,5914
5,slq320___usual_sleep_time_on_weekends,categorical,0.62,55,cramers_v,0.102843,0.102843,1.991802e-01,5912


In [36]:
numeric_features_sf = []
categorical_features_sf = []

for col in sleep_focused_final_features:
    if infer_feature_type(X_train_sf[col]) == "numeric":
        numeric_features_sf.append(col)
    else:
        categorical_features_sf.append(col)

print("Sleep-focused numeric features:", numeric_features_sf)
print()
print("Sleep-focused categorical features:", categorical_features_sf)

Sleep-focused numeric features: ['sld012___sleep_hours___weekdays_or_workdays', 'sld013___sleep_hours___weekends', 'alq170___past_30_days_#_times_4_5_drinks_on_an_oc']

Sleep-focused categorical features: ['slq030___how_often_do_you_snore?', 'slq300___usual_sleep_time_on_weekdays_or_workdays', 'slq310___usual_wake_time_on_weekdays_or_workdays', 'slq320___usual_sleep_time_on_weekends', 'slq330___usual_wake_time_on_weekends', 'dpq040___feeling_tired_or_having_little_energy', 'bpq020___ever_told_you_had_high_blood_pressure', 'cdq010___shortness_of_breath_on_stairs/inclines', 'mcq010___ever_been_told_you_have_asthma', 'mcq160p___ever_told_you_had_copd_emphysema', 'paq605___vigorous_work_activity', 'paq665___moderate_recreational_activities', 'alq111___ever_had_a_drink_of_any_kind_of_alcohol']


In [37]:
preprocessor_sf = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features_sf),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features_sf),
    ],
    remainder="drop"
)

In [38]:
logreg_pipeline_sf = Pipeline(steps=[
    ("preprocessor", preprocessor_sf),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

rf_pipeline_sf = Pipeline(steps=[
    ("preprocessor", preprocessor_sf),
    ("model", RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

In [39]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}

logreg_cv_sf = cross_validate(logreg_pipeline_sf, X_train_sf, y_train_sf, cv=cv, scoring=scoring, n_jobs=-1)
rf_cv_sf = cross_validate(rf_pipeline_sf, X_train_sf, y_train_sf, cv=cv, scoring=scoring, n_jobs=-1)

cv_summary_sf = pd.DataFrame({
    "model": ["logreg_sleep_focused", "random_forest_sleep_focused"],
    "roc_auc_mean": [logreg_cv_sf["test_roc_auc"].mean(), rf_cv_sf["test_roc_auc"].mean()],
    "avg_precision_mean": [logreg_cv_sf["test_avg_precision"].mean(), rf_cv_sf["test_avg_precision"].mean()],
    "f1_mean": [logreg_cv_sf["test_f1"].mean(), rf_cv_sf["test_f1"].mean()],
    "precision_mean": [logreg_cv_sf["test_precision"].mean(), rf_cv_sf["test_precision"].mean()],
    "recall_mean": [logreg_cv_sf["test_recall"].mean(), rf_cv_sf["test_recall"].mean()],
}).sort_values("avg_precision_mean", ascending=False)

cv_summary_sf

,model,roc_auc_mean,avg_precision_mean,f1_mean,precision_mean,recall_mean
1,random_forest_sleep_focused,0.747985,0.598010,0.583016,0.536899,0.637900
0,logreg_sleep_focused,0.741652,0.590595,0.568973,0.522458,0.624892


In [40]:
best_pipeline_sf = rf_pipeline_sf
best_model_name_sf = "rf_sleep_focused"

best_pipeline_sf.fit(X_train_sf, y_train_sf)

test_proba_sf = best_pipeline_sf.predict_proba(X_test_sf)[:, 1]
test_pred_sf = (test_proba_sf >= 0.50).astype(int)

print("Best sleep-focused model:", best_model_name_sf)
print("ROC-AUC:", round(roc_auc_score(y_test_sf, test_proba_sf), 4))
print("Average Precision:", round(average_precision_score(y_test_sf, test_proba_sf), 4))
print("Precision:", round(precision_score(y_test_sf, test_pred_sf, zero_division=0), 4))
print("Recall:", round(recall_score(y_test_sf, test_pred_sf, zero_division=0), 4))
print("F1:", round(f1_score(y_test_sf, test_pred_sf, zero_division=0), 4))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test_sf, test_pred_sf))
print()
print(classification_report(y_test_sf, test_pred_sf, zero_division=0))

Best sleep-focused model: rf_sleep_focused
ROC-AUC: 0.7552
Average Precision: 0.6128
Precision: 0.5328
Recall: 0.6424
F1: 0.5825

Confusion Matrix:
[[736 271]
 [172 309]]

              precision    recall  f1-score   support

           0       0.81      0.73      0.77      1007
           1       0.53      0.64      0.58       481

    accuracy                           0.70      1488
   macro avg       0.67      0.69      0.68      1488
weighted avg       0.72      0.70      0.71      1488



In [41]:
perm_sf = permutation_importance(
    best_pipeline_sf,
    X_test_sf,
    y_test_sf,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring="average_precision",
    n_jobs=-1
)

perm_df_sf = pd.DataFrame({
    "feature": X_test_sf.columns,
    "importance_mean": perm_sf.importances_mean,
    "importance_std": perm_sf.importances_std
}).sort_values("importance_mean", ascending=False)

perm_df_sf

,feature,importance_mean,importance_std
7,dpq040___feeling_tired_or_having_little_energy,0.078600,0.009646
9,cdq010___shortness_of_breath_on_stairs/inclines,0.049629,0.008709
8,bpq020___ever_told_you_had_high_blood_pressure,0.032154,0.008060
0,slq030___how_often_do_you_snore?,0.030458,0.005566
10,mcq010___ever_been_told_you_have_asthma,0.011356,0.003579
11,mcq160p___ever_told_you_had_copd_emphysema,0.009690,0.003151
14,alq111___ever_had_a_drink_of_any_kind_of_alcohol,0.005503,0.002048
1,sld012___sleep_hours___weekdays_or_workdays,0.002738,0.000859
2,sld013___sleep_hours___weekends,0.001605,0.001490
4,slq310___usual_wake_time_on_weekdays_or_workdays,0.001603,0.001485


In [42]:
combined_df_sf = (
    sf_univariate_df.merge(perm_df_sf, on="feature", how="left")
    .merge(sleep_focus_audit_df[["feature", "missing_pct", "n_unique"]], on="feature", how="left")
)

combined_df_sf = combined_df_sf.sort_values(
    ["importance_mean", "abs_association"],
    ascending=[False, False]
)

combined_df_sf

,feature,feature_type,missing_pct_train,n_unique_train,association_metric,association_value,abs_association,p_value,n_train_non_null,importance_mean,importance_std,missing_pct,n_unique
0,dpq040___feeling_tired_or_having_little_energy,categorical,12.81,6,cramers_v,0.291063,0.291063,9.354725e-93,5187,0.078600,0.009646,13.03,6
1,cdq010___shortness_of_breath_on_stairs/inclines,categorical,43.69,3,cramers_v,0.271130,0.271130,3.347455e-54,3350,0.049629,0.008709,43.83,3
2,bpq020___ever_told_you_had_high_blood_pressure,categorical,0.00,3,cramers_v,0.226528,0.226528,5.135945e-67,5949,0.032154,0.008060,0.00,3
3,slq030___how_often_do_you_snore?,categorical,0.00,6,cramers_v,0.215314,0.215314,1.591744e-57,5949,0.030458,0.005566,0.00,6
7,mcq010___ever_been_told_you_have_asthma,categorical,0.00,3,cramers_v,0.144117,0.144117,1.477888e-27,5949,0.011356,0.003579,0.00,3
4,mcq160p___ever_told_you_had_copd_emphysema,categorical,6.44,3,cramers_v,0.181734,0.181734,1.207192e-40,5566,0.009690,0.003151,6.20,3
10,alq111___ever_had_a_drink_of_any_kind_of_alcohol,categorical,12.51,2,cramers_v,0.089879,0.089879,8.909926e-11,5205,0.005503,0.002048,12.75,2
13,sld012___sleep_hours___weekdays_or_workdays,numeric,0.76,24,pointbiserial,-0.029323,0.029323,2.425026e-02,5904,0.002738,0.000859,0.71,24
11,sld013___sleep_hours___weekends,numeric,0.89,24,pointbiserial,-0.089011,0.089011,7.545816e-12,5896,0.001605,0.001490,0.86,24
6,slq310___usual_wake_time_on_weekdays_or_workdays,categorical,0.61,107,cramers_v,0.144723,0.144723,1.134996e-01,5913,0.001603,0.001485,0.62,113


In [43]:
set1_cv = cv_summary.copy()
set2_cv = cv_summary_sf.copy()

comparison_cv = pd.concat([set1_cv, set2_cv], ignore_index=True)
comparison_cv

,model,roc_auc_mean,avg_precision_mean,f1_mean,precision_mean,recall_mean
0,logreg,0.789819,0.656604,0.616223,0.564452,0.678473
1,random_forest,0.794305,0.656453,0.614838,0.589185,0.643090
2,random_forest_sleep_focused,0.747985,0.598010,0.583016,0.536899,0.637900
3,logreg_sleep_focused,0.741652,0.590595,0.568973,0.522458,0.624892


In [44]:
comparison_test = pd.DataFrame([
    {
        "run": "set1_all_questionnaire",
        "model": best_model_name,
        "roc_auc": roc_auc_score(y_test, test_proba),
        "avg_precision": average_precision_score(y_test, test_proba),
        "precision": precision_score(y_test, test_pred, zero_division=0),
        "recall": recall_score(y_test, test_pred, zero_division=0),
        "f1": f1_score(y_test, test_pred, zero_division=0),
        "n_features": len(final_features),
    },
    {
        "run": "set2_sleep_focused",
        "model": best_model_name_sf,
        "roc_auc": roc_auc_score(y_test_sf, test_proba_sf),
        "avg_precision": average_precision_score(y_test_sf, test_proba_sf),
        "precision": precision_score(y_test_sf, test_pred_sf, zero_division=0),
        "recall": recall_score(y_test_sf, test_pred_sf, zero_division=0),
        "f1": f1_score(y_test_sf, test_pred_sf, zero_division=0),
        "n_features": len(sleep_focused_final_features),
    }
])

comparison_test.sort_values("avg_precision", ascending=False)

,run,model,roc_auc,avg_precision,precision,recall,f1,n_features
0,set1_all_questionnaire,logreg,0.787327,0.658227,0.580071,0.677755,0.625120,61
1,set2_sleep_focused,rf_sleep_focused,0.755158,0.612766,0.532759,0.642412,0.582469,16


In [45]:
SF_KEEP_CORE = [
    "slq030___how_often_do_you_snore?",
    "dpq040___feeling_tired_or_having_little_energy",
]

SF_KEEP_OPTIONAL = [
    "sld012___sleep_hours___weekdays_or_workdays",
    "sld013___sleep_hours___weekends",
    "slq300___usual_sleep_time_on_weekdays_or_workdays",
    "slq310___usual_wake_time_on_weekdays_or_workdays",
    "slq320___usual_sleep_time_on_weekends",
    "slq330___usual_wake_time_on_weekends",
    "bpq020___ever_told_you_had_high_blood_pressure",
    "cdq010___shortness_of_breath_on_stairs/inclines",
]

SF_DROP_FOR_QUIZ = [
    "mcq160p___ever_told_you_had_copd_emphysema",
    "mcq010___ever_been_told_you_have_asthma",
]

def label_sf_quiz_status(feature):
    if feature in SF_KEEP_CORE:
        return "keep_core"
    if feature in SF_KEEP_OPTIONAL:
        return "keep_optional"
    if feature in SF_DROP_FOR_QUIZ:
        return "drop_for_quiz"
    return "review"

combined_df_sf["quiz_status"] = combined_df_sf["feature"].apply(label_sf_quiz_status)
combined_df_sf

,feature,feature_type,missing_pct_train,n_unique_train,association_metric,association_value,abs_association,p_value,n_train_non_null,importance_mean,importance_std,missing_pct,n_unique,quiz_status
0,dpq040___feeling_tired_or_having_little_energy,categorical,12.81,6,cramers_v,0.291063,0.291063,9.354725e-93,5187,0.078600,0.009646,13.03,6,keep_core
1,cdq010___shortness_of_breath_on_stairs/inclines,categorical,43.69,3,cramers_v,0.271130,0.271130,3.347455e-54,3350,0.049629,0.008709,43.83,3,keep_optional
2,bpq020___ever_told_you_had_high_blood_pressure,categorical,0.00,3,cramers_v,0.226528,0.226528,5.135945e-67,5949,0.032154,0.008060,0.00,3,keep_optional
3,slq030___how_often_do_you_snore?,categorical,0.00,6,cramers_v,0.215314,0.215314,1.591744e-57,5949,0.030458,0.005566,0.00,6,keep_core
7,mcq010___ever_been_told_you_have_asthma,categorical,0.00,3,cramers_v,0.144117,0.144117,1.477888e-27,5949,0.011356,0.003579,0.00,3,drop_for_quiz
4,mcq160p___ever_told_you_had_copd_emphysema,categorical,6.44,3,cramers_v,0.181734,0.181734,1.207192e-40,5566,0.009690,0.003151,6.20,3,drop_for_quiz
10,alq111___ever_had_a_drink_of_any_kind_of_alcohol,categorical,12.51,2,cramers_v,0.089879,0.089879,8.909926e-11,5205,0.005503,0.002048,12.75,2,review
13,sld012___sleep_hours___weekdays_or_workdays,numeric,0.76,24,pointbiserial,-0.029323,0.029323,2.425026e-02,5904,0.002738,0.000859,0.71,24,keep_optional
11,sld013___sleep_hours___weekends,numeric,0.89,24,pointbiserial,-0.089011,0.089011,7.545816e-12,5896,0.001605,0.001490,0.86,24,keep_optional
6,slq310___usual_wake_time_on_weekdays_or_workdays,categorical,0.61,107,cramers_v,0.144723,0.144723,1.134996e-01,5913,0.001603,0.001485,0.62,113,keep_optional


In [46]:
sf_univariate_path = OUTPUT_DIR / "sleep_focused_univariate_ranking.csv"
sf_perm_path = OUTPUT_DIR / "sleep_focused_permutation_importance.csv"
sf_combined_path = OUTPUT_DIR / "sleep_focused_quiz_candidates.csv"
sf_comparison_path = OUTPUT_DIR / "sleep_focused_vs_set1_comparison.csv"

sf_univariate_df.to_csv(sf_univariate_path, index=False)
perm_df_sf.to_csv(sf_perm_path, index=False)
combined_df_sf.to_csv(sf_combined_path, index=False)
comparison_test.to_csv(sf_comparison_path, index=False)

print("Saved:")
print(sf_univariate_path.resolve())
print(sf_perm_path.resolve())
print(sf_combined_path.resolve())
print(sf_comparison_path.resolve())

Saved:
C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_sleep_disorder\sleep_focused_univariate_ranking.csv
C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_sleep_disorder\sleep_focused_permutation_importance.csv
C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_sleep_disorder\sleep_focused_quiz_candidates.csv
C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_sleep_disorder\sleep_focused_vs_set1_comparison.csv


In [48]:
def evaluate_top_n_features(feature_list, top_n, model_type="rf"):
    selected = feature_list[:top_n]

    num_feats = [c for c in selected if infer_feature_type(X_train[c]) == "numeric"]
    cat_feats = [c for c in selected if infer_feature_type(X_train[c]) == "categorical"]

    local_preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_feats),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), cat_feats),
        ]
    )

    if model_type == "rf":
        model = RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=5,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    else:
        model = LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )

    pipe = Pipeline([
        ("preprocessor", local_preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train[selected], y_train)
    proba = pipe.predict_proba(X_test[selected])[:, 1]
    pred = (proba >= 0.5).astype(int)

    return {
        "top_n": top_n,
        "model_type": model_type,
        "roc_auc": roc_auc_score(y_test, proba),
        "avg_precision": average_precision_score(y_test, proba),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0)
    }

In [49]:
ranked_features_sf = perm_df_sf["feature"].tolist()

topn_results_sf = []
for n in [3, 5, 7, 10]:
    topn_results_sf.append(evaluate_top_n_features(ranked_features_sf, n, model_type="rf"))
    topn_results_sf.append(evaluate_top_n_features(ranked_features_sf, n, model_type="logreg"))

topn_df_sf = pd.DataFrame(topn_results_sf).sort_values(
    ["avg_precision", "roc_auc"],
    ascending=False
)

topn_df_sf

,top_n,model_type,roc_auc,avg_precision,precision,recall,f1
5,7,logreg,0.759921,0.620599,0.536878,0.650728,0.588346
6,10,rf,0.758967,0.616010,0.530680,0.665281,0.590406
3,5,logreg,0.757328,0.615264,0.530100,0.659044,0.587581
7,10,logreg,0.752969,0.614166,0.548214,0.638254,0.589817
4,7,rf,0.750886,0.603426,0.521667,0.650728,0.579093
2,5,rf,0.749379,0.597205,0.522167,0.661123,0.583486
0,3,rf,0.723766,0.562535,0.548515,0.575884,0.561866
1,3,logreg,0.722121,0.561512,0.548515,0.575884,0.561866
